# G01: What Is Mechanistic Interpretability?

**Prerequisites:** Basic understanding of what a transformer is (attention, MLPs, residual connections).

**What you'll learn:**
- What mechanistic interpretability is and why it matters
- The key questions mechinterp tries to answer
- The residual stream as the central organizing concept
- A roadmap of tools and techniques
- Where the field is heading

---

## The Problem of Opacity

A modern language model is, at its core, a massive lookup-free function: hundreds of millions (or billions) of floating-point parameters, composed through layers of matrix multiplications, nonlinearities, and normalization steps, producing text that can be eerily coherent, creative, and knowledgeable. But ask *why* it said what it said, and you hit a wall.

There is no comments section in the weights. No variable named `is_this_safe`. No subroutine labeled `recall_fact_about_paris`. The model's "algorithm" is implicit — spread across billions of parameters in ways that resist casual inspection.

This opacity creates real problems:

- **Safety:** A model might refuse a harmless request about chemistry homework while happily complying with a cleverly rephrased harmful one. Without understanding *how* it makes refusal decisions internally, we're left playing whack-a-mole with prompt injections and jailbreaks.

- **Science:** We've built artifacts that can write poetry, prove theorems, and translate between languages — but we don't really understand what they've learned. That's like having a telescope that shows you galaxies while the lens manufacturer shrugs and says "we're not sure how it works."

- **Engineering:** When a model hallucinates a fake citation or gets a math problem wrong, the fix is usually "add more data" or "fine-tune harder." These are blunt instruments. If we understood *which components* were responsible, we could intervene surgically.

- **Trust:** Deploying opaque systems in high-stakes settings — medicine, law, infrastructure — requires a level of confidence we can't currently provide. "It usually works" is not an engineering standard.

Mechanistic interpretability is the bet that we can do better. That these networks, despite their complexity, implement *understandable algorithms* — and that with the right tools, we can find them.

## What Mechanistic Interpretability Asks

The field is organized around three interlocking questions:

### 1. What algorithms does the model implement?

Language models don't just memorize input-output pairs — they develop internal *procedures*. The most famous example is **induction heads**: a two-layer circuit where one attention head identifies the previous occurrence of the current token, and a second head copies what came *after* that previous occurrence to predict the next token. This is in-context pattern matching, and it emerges reliably across models of different sizes and architectures.

Discovering these algorithms is like finding assembly code in a CPU — you can see the actual computation the model performs, not just its inputs and outputs.

### 2. Which components are responsible?

A transformer has a lot of moving parts: dozens of layers, each containing multiple attention heads and an MLP block. For any given behavior, most of these components are irrelevant. The goal is to narrow down: which specific heads, which specific MLP neurons, are doing the work?

For example, in the **Indirect Object Identification** (IOI) task — predicting "John" in "When Mary and John went to the store, Mary gave a gift to ___" — researchers identified roughly 26 attention heads in GPT-2 Small that form the core circuit, out of 144 total heads. The rest are largely spectators.

### 3. How do components compose?

Individual components rarely act alone. An attention head's output becomes another head's input; an MLP refines what attention has routed. Understanding these compositions — the *circuits* — is where the real explanatory power lies.

In the induction head example, previous-token heads in early layers feed positional information into induction heads in later layers. Without this specific composition, in-context copying fails. The algorithm isn't in any single head — it's in the connection pattern.

### The methodology

The core approach treats neural networks the way electrical engineers treat circuits: identify the components, trace the connections, measure what flows through them, and test your understanding by intervening. If you think head 7.3 is responsible for copying names, you should be able to knock it out and see name-copying break, or amplify it and see name-copying strengthen. This is **causal** interpretability — not just correlation, but verified mechanism.

## The Residual Stream: The Central Organizing Concept

If there's one idea that makes mechanistic interpretability click, it's the **residual stream** picture.

In a transformer, every layer's output is *added* to its input via a residual connection. This means there's a running vector — the residual stream — that accumulates contributions from every component the token passes through:

```
  Token Embedding + Positional Embedding
              │
              ▼
    ┌─────────────────────┐
    │   Residual Stream    │ ◄── This is the "communication bus"
    └──────┬──────────────┘
           │
     ┌─────┴─────┐
     │  Read from │
     │   stream   │
     ▼            ▼
  ┌──────┐   ┌────────┐
  │ Attn │   │  MLP   │
  └──┬───┘   └───┬────┘
     │            │
     │  Write to  │
     │   stream   │
     ▼            ▼
    ┌─────────────────────┐
    │   Residual Stream    │ ◄── Updated with both contributions
    └──────┬──────────────┘
           │
          ...  (repeat for each layer)
           │
           ▼
    ┌─────────────┐
    │  Unembedding │ ──► Logits over vocabulary
    └─────────────┘
```

This picture has profound consequences:

**Everything lives in the same space.** The residual stream at layer 0, layer 6, and layer 11 all occupy the same vector space (typically called `d_model`). This means you can meaningfully compare representations across layers — and you can apply the unembedding matrix to *any* layer's residual stream to see what the model would predict if it had to output right then.

**Components communicate by reading and writing.** Each attention head and MLP block reads from the residual stream (via its input projection), does some computation, and writes the result back (via its output projection). The residual stream is the shared memory that all components access.

**The model's computation is a sum.** Because of the residual connections, the final output is literally the sum of the initial embedding plus every attention and MLP contribution across all layers. This additive structure is what makes techniques like the **logit lens** work — you can project intermediate states through the unembedding and get meaningful predictions, because you're seeing a partial sum of the model's computation.

**Early layers can talk to late layers directly.** Since each layer writes to the same stream, layer 0's output can persist all the way to the final layer without being "overwritten" — later layers add to it, but don't erase it. This is why residual connections are so important: they create a highway for information to flow across arbitrary distances in the network.

## The Tool Inventory

Mechanistic interpretability has developed a rich toolkit over the past few years. Here's a map of the major categories, the IRTK modules that implement them, and the questions they help answer:

| Category | Key IRTK Tools | Question Answered |
|----------|---------------|-------------------|
| **Activation inspection** | `run_with_cache`, `logit_lens`, `residual_stream` | What does the model "think" at each layer? What would it predict if forced to output now? |
| **Attention analysis** | `visualization`, `attention_utils`, `head_analysis` | What information do attention heads route? Which heads are induction heads, previous-token heads, etc.? |
| **Circuit discovery** | `patching`, `circuit_discovery`, `circuits` | Which components implement a given behavior? What's the minimal circuit? |
| **MLP analysis** | `neurons`, `mlp_decomposition`, `knowledge` | How is knowledge stored and retrieved in MLP layers? Which neurons encode which facts? |
| **Feature discovery** | `sae`, `sparse_features`, `transcoder` | What are the natural units of computation? Can we decompose superposed representations? |
| **Causal intervention** | `steering`, `activation_surgery`, `patching` | Does changing internal state X cause output change Y? Is this component necessary and sufficient? |
| **Probing** | `probes`, `activation_probing`, `sparse_probing` | What information is linearly accessible at each layer? When does a concept become decodable? |
| **Weight analysis** | `circuits` (OV/QK), `low_rank`, `weight_structure` | What can the weights compute in principle? What's the effective rank of each head? |
| **Attribution** | `gradients`, `feature_attribution`, `principled_attribution` | Which inputs and components contribute most to a given output? |
| **Comparison** | `model_diff`, `model_comparison`, `cross_model_alignment` | How do two models (or a model before and after fine-tuning) differ internally? |

Don't worry about memorizing this table — the later tutorials in this series will walk through each category hands-on. The point here is that these techniques exist, they're complementary, and together they give you a remarkably detailed picture of what's happening inside a transformer.

## A Preview: The IOI Circuit

To make all of this concrete, let's preview one of the most celebrated results in the field: the **Indirect Object Identification (IOI) circuit**, reverse-engineered in GPT-2 Small by Wang et al. (2023).

### The task

Consider: *"When Mary and John went to the store, Mary gave a gift to ___"*

Humans immediately predict **John** — the person who *hasn't* been mentioned as the subject of "gave." GPT-2 Small does too, with high confidence. But *how*?

### The circuit

Through painstaking activation patching — systematically replacing each component's output with a corrupted version and measuring the effect on the prediction — the researchers identified a circuit of roughly 26 attention heads organized into functional groups:

1. **Duplicate token heads** (early layers): These heads detect that "Mary" appears twice in the sentence. They attend from the second "Mary" back to the first, marking it as repeated.

2. **S-inhibition heads** (middle layers): These heads receive the "this name is repeated" signal and use it to *suppress* "Mary" as a candidate for the indirect object position. They write a negative signal for "Mary" into the residual stream.

3. **Name mover heads** (late layers): These heads attend to all name tokens and copy their identity to the final token position. But because the S-inhibition heads have suppressed "Mary," the name mover heads effectively copy **"John"** — the unsuppressed name.

There are also **backup name mover heads** that activate when the primary name movers are ablated — the model has built-in redundancy.

### Why this matters

This isn't just a cool finding — it's a proof of concept for the entire mechanistic interpretability agenda:

- The model implements a **specific, understandable algorithm** (not just pattern matching on surface statistics).
- The algorithm is **distributed across specific components** that can be individually identified and tested.
- The components **compose in a specific way** — removing any functional group breaks the behavior.
- The understanding is **causal**, not correlational — every claim was verified by intervention.

In **G04**, we'll reproduce this analysis step by step using IRTK.

## The Interpretability Agenda: Where the Field Is Heading

Mechanistic interpretability is a young field — most of the key results are from 2021 onwards — but it's developing fast. Here are the major research directions:

### Mechanistic anomaly detection

If we can characterize a model's *normal* internal computations, we might detect when something unusual is happening — a backdoor trigger activating a trojan circuit, or a model reasoning about deception in its hidden states. This is the interpretability analogue of antivirus software: don't just check inputs and outputs, monitor the internals.

### Scalable oversight

As AI systems become more capable, humans may not be able to evaluate their outputs directly (imagine an AI doing novel mathematical research). Interpretability tools could let us verify that the model is "reasoning correctly" even when we can't check the answer — not by reading every activation, but by monitoring key internal signals like confidence, uncertainty, and whether the model is accessing genuine knowledge or confabulating.

### Feature universality

Do different models learn the same internal representations? Early evidence suggests yes — similar features (like a "base64 detector" or an "is this the indirect object?" feature) show up across models trained independently. If this holds broadly, it would mean interpretability results *transfer*: understand one model, and you've partially understood them all.

### Automated interpretability

Manually reverse-engineering circuits doesn't scale to models with trillions of parameters. A major research direction is using AI systems themselves to interpret other AI systems — automatically labeling neurons, discovering circuits, and generating human-readable explanations. The goal is to make interpretability keep pace with model scaling.

### Superposition and sparse autoencoders

Perhaps the biggest conceptual challenge: models represent far more features than they have dimensions. A 768-dimensional residual stream might encode thousands of distinct concepts by *superposing* them — storing multiple features in overlapping directions. This means individual neurons are **polysemantic** (responding to multiple unrelated concepts), making them hard to interpret.

**Sparse autoencoders (SAEs)** are the leading approach to untangling superposition. By training a wider, sparse model to reconstruct a layer's activations, SAEs learn a dictionary of **monosemantic features** — directions that each correspond to a single interpretable concept. This transforms an opaque 768-dimensional space into a sparse, human-readable feature space that might be thousands of dimensions wide but only has a few active features at any time.

This is arguably the frontier of the field right now: can we build a complete, faithful feature-level description of what a large language model has learned?

## Annotated Bibliography

These are the foundational papers. You don't need to read them all before continuing, but they're here when you want to go deeper.

- **[Elhage et al. 2021 — "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html)**
  The foundational paper. Introduces the residual stream view, attention head decomposition into QK and OV circuits, and virtual weights. If you read one paper, make it this one.

- **[Olsson et al. 2022 — "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895)**
  Discovers induction heads — a two-layer circuit for in-context copying. Shows they appear during a specific phase of training and may drive the sharp decrease in loss known as the "phase change."

- **[Wang et al. 2023 — "Interpretability in the Wild: a Circuit for Indirect Object Identification in GPT-2 Small"](https://arxiv.org/abs/2211.00593)**
  The most detailed circuit analysis to date. Reverse-engineers the IOI circuit, identifying ~26 attention heads organized into functional groups (duplicate token, S-inhibition, name mover, backup).

- **[Bricken et al. 2023 — "Towards Monosemanticity: Decomposing Language Models With Dictionary Learning"](https://transformer-circuits.pub/2023/monosemantic-features/index.html)**
  Trains sparse autoencoders on a one-layer transformer and finds clearly interpretable features. A breakthrough in understanding superposition and polysemanticity.

- **[nostalgebraist 2020 — "interpreting GPT: the logit lens"](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens)**
  The original logit lens blog post. A simple but powerful idea: project intermediate residual stream states through the unembedding matrix to see what each layer "predicts." Surprisingly informative.

- **[Conmy et al. 2023 — "Towards Automated Circuit Discovery for Mechanistic Interpretability"](https://arxiv.org/abs/2304.14997)**
  Introduces ACDC (Automatic Circuit DisCovery): automated methods for finding circuits via iterative edge pruning in the computational graph.

- **[Cunningham et al. 2023 — "Sparse Autoencoders Find Highly Interpretable Features in Language Models"](https://arxiv.org/abs/2309.08600)**
  Scales sparse autoencoders to larger models and demonstrates that the learned features are genuinely interpretable and causally meaningful.

- **[Neel Nanda — "200 Concrete Open Problems in Mechanistic Interpretability"](https://www.alignmentforum.org/s/yivyHaCAmMJ3CqSyj)**
  Not a paper, but an invaluable resource: a curated list of research questions at various difficulty levels. Great for finding a project to work on.

## G-Series Roadmap

This tutorial is the starting point of a guided series that takes you from conceptual understanding to running your own interpretability investigations:

| Tutorial | Topic | What You'll Do |
|----------|-------|----------------|
| **G01** (this one) | What is mechinterp? | Motivation and roadmap |
| **G02** | Your first look inside GPT-2 | Load GPT-2, explore the cache, run the logit lens |
| **G03** | Understanding attention heads | Visualize patterns, classify heads, analyze circuits |
| **G04** | Finding circuits — the IOI case study | Full activation patching investigation |
| **G05** | MLPs as knowledge stores | Neuron analysis, factual recall, polysemanticity |
| **G06** | Sparse autoencoders | Train an SAE, discover features |
| **G07** | Running your own investigation | End-to-end research project on a new question |

Each tutorial builds on the previous ones, but they're also designed to be somewhat self-contained — if you already know the basics, you can jump to the topic that interests you.

**Next: [G02 — Your First Look Inside GPT-2](G02_your_first_look_inside_gpt2.ipynb)**